In [8]:
import requests
import base64
import pandas as pd
import plotly.express as px
import jalali_pandas

In [ ]:
import os

# توکن را در متغیر محیطی MABNA_TOKEN قرار بده (نه داخل کد)
access_token = os.environ["MABNA_TOKEN"]
# Generate Header: "Basic " + Base64(token + ":")
auth_str = f"{access_token}:"
encoded_token = base64.b64encode(auth_str.encode('ascii')).decode('ascii')
headers = {
    "Authorization": f"Basic {encoded_token}",
    "Accept": "application/json"
}

In [11]:
def retrative_read_data(url, params={}):
    page_size = 100  # تعداد رکورد در هر درخواست (بر اساس مثال شما)
    current_skip = 0
    params['_count'] = page_size
    params['_skip'] = current_skip
    data = pd.DataFrame()
    while True:
        params['_skip'] = current_skip
        # print(f"در حال دریافت رکوردهای {current_skip} تا {current_skip + page_size}...")
        df = (pd.DataFrame(requests.get(url, headers=headers, params=params).json()['data']))
        data = pd.concat([data, df])
        
        if len(df) < page_size:
            break
        
        current_skip += page_size
    
    if 'date_time' in data.columns:
        try:
            data['j_date_time'] = data['date_time'].str[:-6].jalali.parse_jalali("%Y%m%d").apply(lambda x: str(x).split(' ')[0])
            data['datetime'] = data['date_time'].str[:-6].jalali.parse_jalali("%Y%m%d").jalali.to_gregorian() 
        except:
            pass 

    return data

In [ ]:
code = {
    "میلگرد": "SteelRebar  ",
    "گندله سنگ آهن": "CD1IOP0001  ",
    "شمش روی": "D1ZNI0001  ",
    "شمش سرب": "LeadIngot   ",
    "گواهی سپرده شمش نقره": "SilverBar   ",
    "مس کاتد": "CopperCthd  ",
    "گواهی سپرده شمش طلا": "GoldBar",
    # دلار کد بورسی ندارد؛ با نام و نوع (currency) پیدا می‌شود
    "دلار": "name:دلار",
}

In [ ]:
import json

BASE = "https://v1.db.api.mabnadp.com"
COLS = ["date_time", "open_price", "high_price", "low_price",
        "close_price", "real_close_price", "close_price_change",
        "real_close_price_change", "volume", "value", "trade_count", "buyer_count"]


def resolve_filter(value):
    """کد بورسی -> instrument.code ؛ 'name:دلار' -> instrument.id از روی نام و نوع currency"""
    value = value.strip()
    if not value.startswith("name:"):
        return {"instrument.code": value}
    nm = value[5:]
    r = requests.get(f"{BASE}/exchange/instruments", headers=headers, timeout=60,
                     params={"name": nm, "type": "currency"})
    r.raise_for_status()
    inst = r.json().get("data", [])
    if not inst:
        raise ValueError(f"نماد ارزی با نام {nm} پیدا نشد")
    if len(inst) > 1:
        print(f"هشدار: {len(inst)} نماد برای {nm} پیدا شد:",
              [(i['id'], i.get('name')) for i in inst], "-> اولی استفاده شد")
    return {"instrument.id": inst[0]["id"]}


def fetch_all_trades(value):
    """کل تاریخچه معاملات روزانه یک نماد (با صفحه‌بندی)"""
    flt = resolve_filter(value)
    rows, skip, size = [], 0, 100
    while True:
        params = {**flt, "_sort": "date_time", "_count": size, "_skip": skip}
        r = requests.get(f"{BASE}/exchange/trades", headers=headers, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("data", [])
        rows += batch
        if len(batch) < size:
            break
        skip += size
    return rows


frames = []
for name, c in code.items():
    c = c.strip()
    rows = fetch_all_trades(c)
    if not rows:
        print(f"{name} ({c}): بدون داده")
        continue
    df = pd.DataFrame(rows)
    df = df[[x for x in COLS if x in df.columns]].copy()
    df["date"] = df["date_time"].str[:8]                       # شمسی YYYYMMdd
    df["j_date"] = df["date"].str[:4] + "/" + df["date"].str[4:6] + "/" + df["date"].str[6:8]
    df["datetime"] = df["date"].jalali.parse_jalali("%Y%m%d").jalali.to_gregorian()
    df.insert(0, "name", name)
    df.insert(1, "code", c)
    frames.append(df)
    print(f"{name} ({c}): {len(df)} روز  [{df['j_date'].iloc[0]} تا {df['j_date'].iloc[-1]}]")

prices = pd.concat(frames, ignore_index=True)
prices.to_csv("daily_prices.csv", index=False, encoding="utf-8-sig")

# JSON به تفکیک نماد: {name: [ {date, close_price, ...}, ... ]}
out = {
    n: g.drop(columns=["name", "datetime", "date_time"]).to_dict("records")
    for n, g in prices.groupby("name", sort=False)
}
with open("daily_prices.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2, default=str)

prices.tail()

In [15]:
retrative_read_data(url = "https://v1.db.api.mabnadp.com/exchange/intradaytrades", params={'id': '1'})

""
